In [1]:
import pandas as pd

df = pd.read_csv('../data/processed/featured_car_data.csv')

X = df.drop(columns=['selling_price'])
y = df['selling_price']

X.shape, y.shape

((15281, 29), (15281,))

In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape

((12224, 29), (3057, 29))

In [3]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

preds = model.predict(X_test)

print('MAE:', mean_absolute_error(y_test, preds))
print('R2:', r2_score(y_test, preds))

MAE: 89783.09086490601
R2: 0.9376104746042234


In [4]:
y.describe()

count    1.528100e+04
mean     7.270277e+05
std      6.281884e+05
min      4.000000e+04
25%      3.800000e+05
50%      5.500000e+05
75%      8.000000e+05
max      5.000000e+06
Name: selling_price, dtype: float64

In [5]:
import pandas as pd

importances = pd.Series(model.feature_importances_, index=X_train.columns)
importances.sort_values(ascending=False).head(15)

max_power                   0.701999
vehicle_age                 0.163983
km_driven                   0.032540
model_freq                  0.025240
engine                      0.023839
mileage                     0.016713
brand_Mercedes-Benz         0.010696
brand_Other                 0.005211
brand_Tata                  0.003060
seller_type_Individual      0.002706
seats                       0.002409
brand_Skoda                 0.002270
transmission_type_Manual    0.002089
fuel_type_Diesel            0.001024
fuel_type_Petrol            0.000973
dtype: float64

In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor

for name, m in [('Linear Regression', LinearRegression()),
                ('Gradient Boosting', GradientBoostingRegressor(random_state=42))]:
    m.fit(X_train, y_train)
    p = m.predict(X_test)
    print(name, '- MAE:', mean_absolute_error(y_test, p), '| R2:', r2_score(y_test, p))

Linear Regression - MAE: 171900.7395101808 | R2: 0.7727568775946356
Gradient Boosting - MAE: 104014.61244437344 | R2: 0.9224353741189179


In [7]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'n_estimators': [200, 300, 500],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

rf = RandomForestRegressor(random_state=42, n_jobs=-1)

search = RandomizedSearchCV(
    rf, param_distributions=param_grid,
    n_iter=20, cv=3, scoring='r2',
    random_state=42, n_jobs=-1, verbose=1
)

search.fit(X_train, y_train)

print('Best params:', search.best_params_)
print('Best CV R2:', search.best_score_)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best params: {'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 20}
Best CV R2: 0.9334993130233838


In [8]:
best_model = search.best_estimator_
preds = best_model.predict(X_test)

print('Tuned MAE:', mean_absolute_error(y_test, preds))
print('Tuned R2:', r2_score(y_test, preds))

Tuned MAE: 87844.43335363938
Tuned R2: 0.9419130981313832


In [10]:
import joblib
import json

joblib.dump(best_model, '../models/best_model.pkl')

metrics = {
    'model': 'RandomForestRegressor',
    'best_params': search.best_params_,
    'MAE': mean_absolute_error(y_test, preds),
    'R2': r2_score(y_test, preds)
}

with open('../models/model_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=4)

print('Saved model and metrics.')

Saved model and metrics.
